# Image Colorization con Transfer Learning
### ResNet18 (Encoder preaddestrato) + Decoder custom — Dataset: COCO 2017
---
**Idea:** La rete prende un'immagine in bianco e nero (canale L) e predice i colori (canali ab) usando lo spazio colore LAB.

**Strategia in 2 fasi:**
- **Fase 1** → Encoder ResNet18 congelato, addestro solo il decoder
- **Fase 2** → Fine-tuning completo con learning rate differenziato

## Installazione librerie

In [ ]:
!pip install kagglehub opencv-python scikit-image torchvision tqdm matplotlib

## Import

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from skimage.color import rgb2lab, lab2rgb
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import kagglehub

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponibile: {torch.cuda.is_available()}")

PyTorch version: 2.10.0+cu128
CUDA disponibile: True


## Download Dataset COCO 2017

Scarichiamo il dataset COCO 2017 tramite `kagglehub`. Questo dataset è molto più vasto e variegato, ideale per modelli di computer vision robusti.

In [ ]:
import kagglehub
import os

# Download del dataset COCO 2017
print("Download COCO 2017 in corso...")
path = kagglehub.dataset_download("awsaf49/coco-2017-dataset")
print(f"Dataset scaricato in: {path}")

# Esploriamo la struttura per trovare le immagini
# Solitamente COCO ha sottocartelle come train2017, val2017
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    if level > 1: continue
    img_count = len([f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if img_count > 0 or dirs:
        print(f"{'  ' * level}{os.path.basename(root)}/ ({img_count} immagini)")

Download COCO 2017 in corso...
Using Colab cache for faster access to the 'coco-2017-dataset' dataset.
Dataset scaricato in: /kaggle/input/coco-2017-dataset
coco-2017-dataset/ (0 immagini)
  coco2017/ (0 immagini)


## Raccoglie tutti i path delle immagini

In [ ]:
# Configurazione aggiornata per GAN + U-Net
img_dir = os.path.join(path, 'train2017')
if not os.path.exists(img_dir): img_dir = path

all_images = []
for root, _, files in os.walk(img_dir):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')): all_images.append(os.path.join(root, f))

all_images.sort()

MAX_IMAGES = 15000
IMAGE_SIZE = 256
BATCH_SIZE = 16

import numpy as np
np.random.seed(42)
if len(all_images) > MAX_IMAGES:
    all_images = list(np.random.choice(all_images, MAX_IMAGES, replace=False))

print(f"Immagini selezionate per GAN: {len(all_images)}")

Immagini selezionate per GAN: 15000


## Dataset personalizzato

Uguale al concetto di `gestione_dati.ipynb`, ma invece di restituire `(immagine, classe)`,
restituiamo `(canale_L, canali_ab)`. Il target lo creiamo noi in automatico convertendo in LAB.

In [ ]:
from torchvision import transforms

class ColorizationDataset(Dataset):
    def __init__(self, image_paths, size=256):
        self.paths = image_paths
        self.size  = size
        # Aggiungiamo trasformazioni per combattere l'effetto seppia
        self.transforms = transforms.Compose([
            transforms.Resize((size, size), Image.BICUBIC),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.5),
            transforms.RandomHorizontalFlip(),
        ])
        print(f" Dataset creato: {len(self.paths)} immagini con ColorJitter")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert('RGB')
            # Applichiamo le trasformazioni
            img = self.transforms(img)

            img_np = np.array(img, dtype=np.float32) / 255.0
            lab = rgb2lab(img_np).astype(np.float32)

            # L: 0-100 -> Normalizzato tra -1 e 1
            L = (lab[:, :, 0] / 50.0) - 1.0
            L = torch.tensor(L).unsqueeze(0)

            # ab: -128, 127 -> Normalizzato tra -1 e 1
            ab = lab[:, :, 1:] / 128.0
            ab = torch.tensor(ab).permute(2, 0, 1)

            return L, ab
        except Exception:
            return self.__getitem__((idx + 1) % len(self.paths))

## DataLoader (Train / Validation split)

In [ ]:
# Split 80% train, 20% validation
np.random.seed(42)
np.random.shuffle(all_images)

split      = int(0.8 * len(all_images))
train_imgs = all_images[:split]
val_imgs   = all_images[split:]

train_set = ColorizationDataset(train_imgs, size=IMAGE_SIZE)
val_set   = ColorizationDataset(val_imgs,   size=IMAGE_SIZE)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f" Train: {len(train_set)} immagini | Validation: {len(val_set)} immagini")
print(f" Batch size: {BATCH_SIZE} | Batch per epoch: {len(train_loader)}")

 Dataset creato: 12000 immagini con ColorJitter
 Dataset creato: 3000 immagini con ColorJitter
 Train: 12000 immagini | Validation: 3000 immagini
 Batch size: 16 | Batch per epoch: 750


## Verifica visiva del dataset

Controlliamo che i dati vengano caricati e convertiti correttamente prima del training.

In [ ]:
def lab_to_rgb(L_tensor, ab_tensor, saturation_boost=1.5):

    L_np = (L_tensor.squeeze().cpu().numpy() + 1.0) * 50.0
    # Applichiamo il boost di saturazione ai canali ab
    ab_np = ab_tensor.permute(1, 2, 0).cpu().numpy() * 128.0 * saturation_boost

    # Clip dei valori per evitare che escano dal range LAB standard
    ab_np = np.clip(ab_np, -128, 127)

    lab = np.stack([L_np, ab_np[:,:,0], ab_np[:,:,1]], axis=2)
    return np.clip(lab2rgb(lab), 0, 1)

## Architettura del modello

**ResNet18** come encoder preaddestrato + **Decoder custom** con blocchi ConvTranspose2d.

- L'encoder capisce cosa c'è nell'immagine (sfrutta il pretraining su ImageNet)
- Il decoder impara a rimettere i colori
- `congela_encoder()` / `scongela_encoder()` gestiscono le due fasi di training

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.block(x)

class ColorizerResNet(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights='DEFAULT')
        self.enc1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu) # 64
        self.enc2 = nn.Sequential(resnet.maxpool, resnet.layer1)         # 64
        self.enc3 = resnet.layer2                                         # 128
        self.enc4 = resnet.layer3                                         # 256
        self.dec4 = DecoderBlock(256, 128)
        self.dec3 = DecoderBlock(128 + 128, 64) # Skip from enc3
        self.dec2 = DecoderBlock(64 + 64, 64)   # Skip from enc2
        self.dec1 = DecoderBlock(64 + 64, 32)   # Skip from enc1
        self.out = nn.Sequential(nn.Conv2d(32, 2, kernel_size=1), nn.Tanh())

    def forward(self, x):
        x = x.repeat(1, 3, 1, 1)
        e1 = self.enc1(x); e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3)
        d4 = self.dec4(e4)
        d3 = self.dec3(torch.cat([d4, e3], dim=1))
        d2 = self.dec2(torch.cat([d3, e2], dim=1))
        d1 = self.dec1(torch.cat([d2, e1], dim=1))
        return self.out(d1)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        def d_block(in_f, out_f, stride=2): return nn.Sequential(nn.Conv2d(in_f, out_f, 4, stride, 1), nn.BatchNorm2d(out_f), nn.LeakyReLU(0.2, inplace=True))
        self.model = nn.Sequential(d_block(3, 64), d_block(64, 128), d_block(128, 256), d_block(256, 512, stride=1), nn.Conv2d(512, 1, 4, 1, 1))
    def forward(self, x): return self.model(x)

## Setup device e funzioni training

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"  GPU:  {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


def train_epoch(model, loader, criterion, optimizer, device):
    """Esegue un'epoca di training, restituisce la loss media."""
    model.train()
    running_loss = 0.0
    for L, ab in tqdm(loader, desc='  Training', leave=False):
        L, ab = L.to(device), ab.to(device)
        optimizer.zero_grad()
        loss = criterion(model(L), ab)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(loader)


def val_epoch(model, loader, criterion, device):
    """Esegue un'epoca di validation, restituisce la loss media."""
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for L, ab in tqdm(loader, desc='  Validation', leave=False):
            L, ab = L.to(device), ab.to(device)
            running_loss += criterion(model(L), ab).item()
    return running_loss / len(loader)


def salva_checkpoint(model, path, info=''):
    torch.save(model.state_dict(), path)
    print(f"Salvato -> {path}  {info}")

Device: cuda
  GPU:  Tesla T4
  VRAM: 15.6 GB


## FASE 1: Encoder congelato

Addestriamo **solo il decoder**. L'encoder ResNet18 non viene toccato.
Questo è molto più veloce e insegna al decoder a usare le feature già estratte da ResNet.

In [ ]:
EPOCHS = 20
LR = 2e-4
net_G = ColorizerResNet().to(device)
net_D = Discriminator().to(device)
GAN_criterion = nn.BCEWithLogitsLoss(); L1_criterion = nn.L1Loss()
opt_G = torch.optim.Adam(net_G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(net_D.parameters(), lr=LR, betas=(0.5, 0.999))

print(f"Inizio training GAN + U-Net per {EPOCHS} epoche...")
for epoch in range(1, EPOCHS + 1):
    net_G.train(); net_D.train()
    for L, ab in tqdm(train_loader, desc=f'Epoch {epoch}'):
        L, ab = L.to(device), ab.to(device)
        # Train D
        opt_D.zero_grad()
        fake_ab = net_G(L); fake_img = torch.cat([L, fake_ab], dim=1); real_img = torch.cat([L, ab], dim=1)
        pred_fake = net_D(fake_img.detach()); loss_D_fake = GAN_criterion(pred_fake, torch.zeros_like(pred_fake))
        pred_real = net_D(real_img); loss_D_real = GAN_criterion(pred_real, torch.ones_like(pred_real))
        loss_D = (loss_D_fake + loss_D_real) * 0.5; loss_D.backward(); opt_D.step()
        # Train G
        opt_G.zero_grad()
        pred_fake_G = net_D(fake_img); loss_G_GAN = GAN_criterion(pred_fake_G, torch.ones_like(pred_fake_G))
        loss_G_L1 = L1_criterion(fake_ab, ab) * 100
        loss_G = loss_G_GAN + loss_G_L1; loss_G.backward(); opt_G.step()
    torch.save(net_G.state_dict(), 'colorizer_gan_latest.pth')

Inizio training GAN + U-Net per 20 epoche...


Epoch 18:  28%|██▊       | 208/750 [01:49<04:47,  1.89it/s]

## Grafico Fase 1

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(train_losses_f1, label='Train Loss', color='steelblue', linewidth=2)
plt.plot(val_losses_f1,   label='Val Loss',   color='orangered', linewidth=2, linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Fase 1 — Training Decoder (Encoder congelato)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## FASE 2: Fine-tuning completo

Scongela tutto il modello. L'encoder viene aggiornato con un **learning rate molto basso** (1e-5) per evitare il **catastrophic forgetting** — cioè che la rete dimentichi tutto quello che ResNet18 sapeva già su milioni di immagini ImageNet.

In [ ]:
# ==========================================
# IMPOSTAZIONI FASE 2
# ==========================================
EPOCHS_FASE2 = 10
LR_ENCODER   = 1e-5   # molto basso: non distruggiamo ResNet
LR_DECODER   = 1e-4   # normale per il decoder
# ==========================================

# Carica il miglior modello della Fase 1
model.load_state_dict(torch.load('colorizer_fase1_best.pth', map_location=device))
model.scongela_encoder()

# Usiamo SmoothL1Loss anche qui per coerenza e saturazione migliore
criterion = nn.SmoothL1Loss()

# Learning rate differenziato: encoder lentissimo, decoder normale
optimizer_fase2 = torch.optim.Adam([
    {'params': model.enc1.parameters(), 'lr': LR_ENCODER},
    {'params': model.enc2.parameters(), 'lr': LR_ENCODER},
    {'params': model.enc3.parameters(), 'lr': LR_ENCODER},
    {'params': model.enc4.parameters(), 'lr': LR_ENCODER},
    {'params': model.dec4.parameters(), 'lr': LR_DECODER},
    {'params': model.dec3.parameters(), 'lr': LR_DECODER},
    {'params': model.dec2.parameters(), 'lr': LR_DECODER},
    {'params': model.dec1.parameters(), 'lr': LR_DECODER},
    {'params': model.out.parameters(),  'lr': LR_DECODER},
])

# Scheduler: dimezza il LR se la val loss non migliora per 3 epoche
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_fase2, mode='min', factor=0.5, patience=3
)

train_losses_f2, val_losses_f2 = [], []
best_val_loss_f2 = float('inf')

print(f"\nFASE 2 — Fine-tuning completo ({EPOCHS_FASE2} epoche)")
print('-' * 55)

for epoch in range(1, EPOCHS_FASE2 + 1):
    train_loss = train_epoch(model, train_loader, criterion, optimizer_fase2, device)
    val_loss   = val_epoch(model, val_loader, criterion, device)

    train_losses_f2.append(train_loss)
    val_losses_f2.append(val_loss)

    scheduler.step(val_loss)

    if val_loss < best_val_loss_f2:
        best_val_loss_f2 = val_loss
        salva_checkpoint(model, 'colorizer_finale.pth', f'<- migliore (val={val_loss:.4f})')

    print(f"Epoch {epoch:02d}/{EPOCHS_FASE2} | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

print('-' * 55)
print(f"Fase 2 completata. Miglior val loss: {best_val_loss_f2:.4f}")
print("Modello finale salvato in: colorizer_finale.pth")

## Grafico completo (Fase 1 + Fase 2)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses_f1, label='Train', color='steelblue', linewidth=2)
ax1.plot(val_losses_f1,   label='Val',   color='orangered', linewidth=2, linestyle='--')
ax1.set_title('Fase 1 — Solo Decoder (Encoder congelato)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(train_losses_f2, label='Train', color='steelblue', linewidth=2)
ax2.plot(val_losses_f2,   label='Val',   color='orangered', linewidth=2, linestyle='--')
ax2.set_title('Fase 2 — Fine-tuning Completo')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MSE Loss')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('Andamento Training — Image Colorization (Mirflickr25k)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafico salvato in: training_loss.png")

## Test visivo sul validation set

Mostra il confronto fianco a fianco: **B&N input | Predetto dal modello | Originale**.

In [ ]:
def mostra_risultati(model, dataset, n=6):
    model.eval()
    indices = np.random.choice(len(dataset), n, replace=False)

    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    fig.suptitle('Risultati Colorizzazione — COCO\nB&N Input  |  Predetto dal Modello  |  Originale',
                 fontsize=13, fontweight='bold')

    for ax, title in zip(axes[0], ['Input (B&N)', 'Predetto', 'Originale']):
        ax.set_title(title, fontweight='bold')

    for row, idx in enumerate(indices):
        L, ab_true = dataset[int(idx)]
        with torch.no_grad():
            ab_pred = model(L.unsqueeze(0).to(device)).squeeze(0).cpu()

        axes[row, 0].imshow(L.squeeze().numpy(), cmap='gray')
        axes[row, 1].imshow(lab_to_rgb(L, ab_pred))
        axes[row, 2].imshow(lab_to_rgb(L, ab_true))
        for ax in axes[row]: ax.axis('off')

    plt.tight_layout()
    plt.savefig('risultati_colorization.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Grafico salvato in: risultati_colorization.png")


model.load_state_dict(torch.load('colorizer_finale.pth', map_location=device))
mostra_risultati(model, val_set, n=6)

## Salvataggio finale e riepilogo

In [ ]:
torch.save(model.state_dict(), 'colorizer_finale.pth')

size_mb = os.path.getsize('colorizer_finale.pth') / 1e6
print(f"Pesi salvati -> colorizer_finale.pth  ({size_mb:.1f} MB)")
print("Copia questo file nella cartella del backend FastAPI per il sito web\n")

print("=" * 58)
print("RIEPILOGO PROGETTO")
print("=" * 58)
print(f"Dataset:          Mirflickr25k ({len(all_images)} immagini usate)")
print(f"Spazio colore:    LAB  (L=input B&N, ab=target colori)")
print(f"Architettura:     ResNet18 (encoder) + Decoder custom")
print(f"Loss function:    MSELoss")
print(f"Optimizer:        Adam con LR differenziato encoder/decoder")
print(f"Scheduler:        ReduceLROnPlateau (patience=3)")
print(f"Fase 1:           {EPOCHS_FASE1} epoche — solo decoder (encoder congelato)")
print(f"Fase 2:           {EPOCHS_FASE2} epoche — fine-tuning completo")
print(f"Train images:     {len(train_set)}")
print(f"Val images:       {len(val_set)}")
print(f"Best val loss:    {best_val_loss_f2:.4f}")
print("=" * 58)